# 01 · Exploratory data analysis

**Black Box → Actionable Insight.** Four public datasets, one framing: each is a
consequential *reject* decision a person would want overturned, and the label is
the outcome a counterfactual would have to flip.

| dataset | decision | flip target |
|---|---|---|
| **Adult** (primary) | income ≤ 50K | earn > 50K |
| **German credit** | bad credit risk | good risk |
| **HELOC** | line declined | line granted |
| **Diabetes** | positive screen | no diabetes |

Nothing here touches a model — this is the raw material the black box, the SCM
and the cost model are all built on. Everything loads through the pipeline's own
`load_dataset`, so a notebook and an experiment see identical splits.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
ROOT = _here if (_here / "src").exists() else _here.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

RESULTS = ROOT / "results"

def show_image(path):
    """Display a saved figure inline in a notebook; no-op-print in a script."""
    path = pathlib.Path(path)
    try:
        from IPython.display import Image, display  # noqa: PLC0415
        display(Image(filename=str(path)))
    except Exception:
        print("figure saved at", path)

def read_result(name):
    """Read a results CSV, or say which experiment writes it."""
    p = RESULTS / f"{name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"{p} missing - run the experiment that writes it")
    return pd.read_csv(p)

In [ ]:
from src.config import DATASETS
from src.data import load_dataset

bundles = {d: load_dataset(d) for d in DATASETS}
summary = pd.DataFrame([
    {"dataset": d,
     "rows": b.frame.shape[0],
     "features": b.X.shape[1],
     "train": len(b.train_idx),
     "test": len(b.test_idx),
     "positive rate": round(float((b.y == 1).mean()), 3),
     "flip target": b.spec.positive_meaning}
    for d, b in bundles.items()
]).set_index("dataset")
summary

## The feature taxonomy is the whole game

Recourse only means anything once features are sorted into what a person **can**
change, what they **cannot**, and what is **protected**. The cost model and the
SCM both read this taxonomy; a plan that asks someone to change their age or race
is not cheap, it is *inadmissible*.

In [ ]:
b = bundles["adult"]
feat = pd.DataFrame([
    {"feature": f.name, "kind": f.kind, "unit": f.unit,
     "protected": f.protected, "manipulable": f.manipulable,
     "monotone": f.monotone}
    for f in b.spec.features
])
feat

In [ ]:
# Class balance across the four datasets, saved through the paper's style.
from src.utils.io import save_figure
from src.viz import figure, label_of, ACCENT, WARN

fig, axes = figure(1, len(DATASETS), figsize=(3.0 * len(DATASETS), 2.8),
                   squeeze=False)
for ax, d in zip(axes.ravel(), DATASETS):
    y = bundles[d].y
    ax.bar(["rejected\n(flip from)", "accepted\n(flip to)"],
           [int((y == 0).sum()), int((y == 1).sum())], color=[WARN, ACCENT])
    ax.set_title(label_of(d))
fig.suptitle("Class balance: the minority class is the one recourse serves", y=1.02)
paths = save_figure(fig, "eda_class_balance", subdir="eda")
show_image(paths[0])

## Deviation — the diabetes dataset

Diabetes was added as a fourth dataset. Two things were done to it, both recorded
in the README's deviations:

1. **Label inverted** to match the others' "reject to overturn" framing:
   `no_diabetes = 1 - Outcome`, so the positive class is the good outcome a plan
   flips *to*.
2. **A class-conditional-imputation leak was found and removed** — zeros in
   `Glucose`, `BloodPressure`, etc. had been imputed with the class mean, which
   leaks the label into the features. They are treated as missing without peeking
   at the outcome.

In [ ]:
d = bundles["diabetes"]
print("diabetes target:", d.spec.target,
      "| positive =", d.spec.positive_meaning)
print("balance:", dict(pd.Series(d.y).value_counts().sort_index()))
d.frame.describe().T[["mean", "std", "min", "max"]].round(2)